# Data Quality and Preprocessing

This notebook documents the data-quality and preprocessing pipeline developed
for the Campania Financial & ESG Forecasting project.

The original dataset contains proprietary company-level information and
cannot be redistributed. Therefore, this public notebook uses a small
synthetic dataset that reproduces the main structural and data-quality
challenges of the original data.

## Objectives

The preprocessing pipeline addresses:

- inconsistent missing-value labels;
- economically implausible values;
- missing temporal observations;
- extreme values and outliers;
- inconsistent column names;
- company-level missingness;
- preparation of clean, model-ready time-series features.

## Original project scale

- 12,422 private companies;
- 117 initial features;
- financial indicators covering 2015–2024;
- ESG indicators covering 2019–2021;
- 7,660 companies retained in the final modelling dataset.

> The synthetic records used below do not represent real companies.

In [ ]:
from __future__ import annotations

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.2f}".format)

RANDOM_STATE = 42

## 1. Synthetic demonstration dataset

The following dataset reproduces selected characteristics of the original
company-level data, including missing values, placeholder strings, temporal
gaps and financially implausible observations.

In [ ]:
raw_data = pd.DataFrame(
    {
        "company_id": [
            "SYN_001",
            "SYN_002",
            "SYN_003",
            "SYN_004",
            "SYN_005",
            "SYN_006",
        ],
        "ateco_code": [10, 47, 62, 41, 49, 86],
        "employees_2024": [7, 28, 64, 12, 0, 105],
        "sales_revenue_eur_2022": [
            850_000,
            1_200_000,
            "n.d.",
            560_000,
            0,
            4_500_000,
        ],
        "sales_revenue_eur_2023": [
            910_000,
            np.nan,
            2_400_000,
            -50_000,
            730_000,
            4_850_000,
        ],
        "sales_revenue_eur_2024": [
            990_000,
            1_450_000,
            2_650_000,
            640_000,
            790_000,
            5_100_000,
        ],
        "ebitda_eur_2022": [
            92_000,
            135_000,
            310_000,
            -15_000,
            0,
            620_000,
        ],
        "ebitda_eur_2023": [
            105_000,
            np.nan,
            355_000,
            8_000,
            -22_000,
            670_000,
        ],
        "ebitda_eur_2024": [
            118_000,
            165_000,
            390_000,
            21_000,
            15_000,
            710_000,
        ],
        "esg_score_2019": [0.31, "n.s.", 0.68, np.nan, 0.22, 0.81],
        "esg_score_2020": [0.35, 0.41, 0.72, 0.29, np.nan, 0.87],
        "esg_score_2021": [0.38, 0.46, 1.35, 0.33, 0.27, 0.91],
    }
)

raw_data

## 2. Missing-value standardization

The original data contained non-standard missing-value labels such as
`n.d.` and `n.s.`. These values are converted to `NaN` before numerical
processing.

In [ ]:
MISSING_LABELS = {
    "n.d.": np.nan,
    "n.s.": np.nan,
    "N.D.": np.nan,
    "N.S.": np.nan,
    "": np.nan,
}

df = raw_data.replace(MISSING_LABELS).copy()

numeric_columns = [
    column
    for column in df.columns
    if column not in {"company_id"}
]

for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

df

## 3. Robust temporal feature selection

Columns are selected by semantic prefix and year rather than by numerical
position. This makes the pipeline less fragile when the dataset structure
changes.

In [ ]:
def select_yearly_columns(
    dataframe: pd.DataFrame,
    prefix: str,
    start_year: int,
    end_year: int,
) -> list[str]:
    """Return existing columns matching a variable prefix and year range."""
    return [
        f"{prefix}_{year}"
        for year in range(start_year, end_year + 1)
        if f"{prefix}_{year}" in dataframe.columns
    ]


sales_columns = select_yearly_columns(
    df,
    prefix="sales_revenue_eur",
    start_year=2022,
    end_year=2024,
)

ebitda_columns = select_yearly_columns(
    df,
    prefix="ebitda_eur",
    start_year=2022,
    end_year=2024,
)

esg_columns = select_yearly_columns(
    df,
    prefix="esg_score",
    start_year=2019,
    end_year=2021,
)

sales_columns, ebitda_columns, esg_columns

## 4. Financial plausibility checks

Zero and negative values must be interpreted according to the economic
meaning of each variable.

For example:

- Sales revenue should normally be strictly positive for active companies.
- EBITDA can legitimately be zero or negative.
- Net income and equity can also be negative.
- An ESG score outside its valid range requires further treatment.

In [ ]:
def replace_non_positive_with_nan(
    dataframe: pd.DataFrame,
    columns: list[str],
) -> pd.DataFrame:
    """Replace zero or negative values in strictly positive variables."""
    result = dataframe.copy()

    for column in columns:
        result.loc[result[column] <= 0, column] = np.nan

    return result


df = replace_non_positive_with_nan(df, sales_columns)

df[
    ["company_id", *sales_columns]
]

## 5. ESG outlier treatment

ESG scores are expected to lie within a normalized range. Extreme values are
capped using winsorization rather than automatically deleting the entire
company record.

In [ ]:
def winsorize_upper_tail(
    dataframe: pd.DataFrame,
    columns: list[str],
    upper_quantile: float = 0.95,
) -> pd.DataFrame:
    """Cap values above the selected column-specific quantile."""
    result = dataframe.copy()

    for column in columns:
        upper_bound = result[column].quantile(upper_quantile)
        result[column] = result[column].clip(upper=upper_bound)

    return result


df = winsorize_upper_tail(
    df,
    columns=esg_columns,
    upper_quantile=0.95,
)

df[
    ["company_id", *esg_columns]
]

## 6. Company-level missingness

Companies with excessive missingness may not contain enough historical
information for reliable forecasting.

The original project excluded companies with more than 40% missing values
across the relevant modelling features.

In [ ]:
model_features = sales_columns + ebitda_columns + esg_columns

df["missing_share"] = df[model_features].isna().mean(axis=1)

missingness_summary = df[
    ["company_id", "missing_share"]
].sort_values("missing_share", ascending=False)

missingness_summary

In [ ]:
MAX_MISSING_SHARE = 0.40

df_filtered = df.loc[
    df["missing_share"] <= MAX_MISSING_SHARE
].copy()

print(f"Companies before filtering: {len(df):,}")
print(f"Companies after filtering:  {len(df_filtered):,}")

## 7. Temporal missing-value imputation

Residual gaps are treated along the temporal dimension of each indicator.

The procedure:

1. interpolates internal gaps linearly;
2. uses the closest available observation for leading or trailing gaps;
3. operates independently for each company and indicator.

In [ ]:
def interpolate_temporal_block(
    dataframe: pd.DataFrame,
    columns: list[str],
) -> pd.DataFrame:
    """Interpolate missing values horizontally across ordered years."""
    result = dataframe.copy()

    result[columns] = (
        result[columns]
        .interpolate(
            axis=1,
            method="linear",
            limit_direction="both",
        )
    )

    return result


df_imputed = df_filtered.copy()

for temporal_block in [
    sales_columns,
    ebitda_columns,
    esg_columns,
]:
    df_imputed = interpolate_temporal_block(
        df_imputed,
        temporal_block,
    )

df_imputed[
    ["company_id", *model_features]
]

## 8. Data-quality report

The final report compares missingness before and after the preprocessing
pipeline.

In [ ]:
quality_report = pd.DataFrame(
    {
        "feature": model_features,
        "missing_before": [
            raw_data[column]
            .replace(MISSING_LABELS)
            .isna()
            .sum()
            for column in model_features
        ],
        "missing_after": [
            df_imputed[column].isna().sum()
            for column in model_features
        ],
    }
)

quality_report["missing_reduction"] = (
    quality_report["missing_before"]
    - quality_report["missing_after"]
)

quality_report

## Key Takeaways

This notebook demonstrates a reusable preprocessing workflow for
heterogeneous company-level data.

The main design principles are:

- interpret values according to their financial meaning;
- avoid relying on fixed column positions;
- treat missingness at both record and temporal levels;
- preserve legitimate negative financial observations;
- avoid exposing proprietary company information;
- separate reusable logic from exploratory analysis.

## Next Step

The next notebook applies deterministic business segmentation based on:

- company size;
- ATECO economic activity;
- economic macro-sector;
- ESG rating;
- employment-growth dynamics.